In [31]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

In [32]:
# 1. Cargar sus datos (cambia 'tus_datos.csv' por el nombre de tu archivo)
df_marketing = pd.read_csv(r"C:\Users\ramir\OneDrive\Escritorio\Simulador\Repositorio\ProjecteData\Equip_30\Data\06-01-2026\06-01-2026_Clean.csv")

In [45]:
df_marketing["perfil_deuda"] = np.where(
    (df_marketing["housing"] == 1) | (df_marketing["loan"] == 1), 1, 0
)

In [46]:
display(df_marketing)

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,perfil_deuda
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_campaign,1,1
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_campaign,1,0
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_campaign,1,1
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_campaign,1,1
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_campaign,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10982,10983,40,management,married,secondary,0,8486,0,0,unknown,6,may,260,3,-1,0,no_campaign,0,0
10983,10984,53,management,married,tertiary,0,20772,0,0,cellular,4,feb,715,1,-1,0,no_campaign,0,0
10984,10985,55,blue-collar,married,primary,0,3297,1,1,telephone,30,apr,96,1,-1,0,no_campaign,0,1
10985,10986,41,management,married,tertiary,0,9,1,0,cellular,22,jul,82,3,-1,0,no_campaign,0,1


In [47]:
df_marketing.dtypes

id               int64
age              int64
job             object
marital         object
education       object
default          int64
balance          int64
housing          int64
loan             int64
contact         object
day              int64
month           object
duration         int64
campaign         int64
pdays            int64
previous         int64
poutcome        object
deposit          int64
perfil_deuda     int64
dtype: object

In [48]:
# 1. Definir los productos y los canales
productos = ['housing', 'loan', 'deposit']
canales = ['telephone', 'cellular', 'unknown']

filas = []

# 2. Calcular datos para cada producto
for prod in productos:
    # Filtrar solo los clientes que sí tienen el producto
    df_prod = df_marketing[df_marketing[prod] == 1]
    
    for canal in canales:
        # Filtrar por canal de contacto
        df_canal = df_prod[df_prod['contact'] == canal]
        
        # Contar clientes y sumar el total de contactos (columna campaign)
        num_clientes = len(df_canal)
        total_contactos = df_canal['campaign'].sum()
        
        filas.append({
            'Producto / Estado': prod.capitalize(),
            'Canal de Contacto': canal.capitalize(),
            'Cantidad de Clientes': num_clientes,
            'Total de Contactos': total_contactos
        })

# 3. Calcular datos para los que NO tienen ningún producto
df_ninguno = df_marketing[(df_marketing['housing'] == 0) & 
                          (df_marketing['loan'] == 0) & 
                          (df_marketing['deposit'] == 0)]

for canal in canales:
    df_canal = df_ninguno[df_ninguno['contact'] == canal]
    num_clientes = len(df_canal)
    total_contactos = df_canal['campaign'].sum()
    
    filas.append({
        'Producto / Estado': 'Ninguno de los tres',
        'Canal de Contacto': canal.capitalize(),
        'Cantidad de Clientes': num_clientes,
        'Total de Contactos': total_contactos
    })

# 4. Crear DataFrame y establecer el índice jerárquico
tabla_final = pd.DataFrame(filas)
tabla_final.set_index(['Producto / Estado', 'Canal de Contacto'], inplace=True)

display(tabla_final)


Cantidad de Clientes  \
Producto / Estado   Canal de Contacto                         
Housing             Telephone                           239   
                    Cellular                           3291   
                    Unknown                            1653   
Loan                Telephone                            80   
                    Cellular                           1031   
                    Unknown                             320   
Deposit             Telephone                           390   
                    Cellular                           4369   
                    Unknown                             530   
Ninguno de los tres Telephone                           182   
                    Cellular                           1454   
                    Unknown                             420   

                                       Total de Contactos  
Producto / Estado   Canal de Contacto                      
Housing             Telephone                         888  
                    Cellular                         7556  
                    Unknown                          4662  
Loan                Telephone                         330  
                    Cellular                         2670  
                    Unknown                           939  
Deposit             Telephone                         935  
                    Cellular                         9077  
                    Unknown                          1312  
Ninguno de los tres Telephone                         539  
                    Cellular                         4432  
                    Unknown                          1157

In [49]:
# 3. Revisamos el cruce de las variables principales para asegurar volumen de datos
tabla_cruce = pd.crosstab(df_marketing['contact'], df_marketing['deposit'])
print("--- Validación de Tamaño de Muestra ---")
print(tabla_cruce)


--- Validación de Tamaño de Muestra ---
deposit       0     1
contact              
cellular   3559  4369
telephone   374   390
unknown    1765   530


In [51]:
# 4. Ajustamos el modelo incluyendo la variable de interés y las 3 de control seleccionadas
# Nota: poutcome y housing se marcan como categorías C()
formula = "deposit ~ C(contact) + age + C(perfil_deuda) + C(poutcome) + campaign"
modelo = smf.logit(formula, data=df_marketing).fit()

# Ver el resumen estadístico completo (opcional)
print(modelo.summary())

Optimization terminated successfully.
         Current function value: 0.602251
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                deposit   No. Observations:                10987
Model:                          Logit   Df Residuals:                    10978
Method:                           MLE   Df Model:                            8
Date:                Thu, 04 Jun 2026   Pseudo R-squ.:                  0.1303
Time:                        10:11:16   Log-Likelihood:                -6616.9
converged:                       True   LL-Null:                       -7608.0
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                      0.7835      0.104      7.568      0.000      

In [ ]:
# 5. Extraemos los Odds Ratios y sus P-valores para medir la significancia estadística
resultados = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo.params),
    'P-valor': modelo.pvalues
})

print("\n--- RESULTADOS FINALES DEL MODELO ---")
print(resultados.round(4))



--- RESULTADOS FINALES DEL MODELO ---
                            Odds Ratio (OR)  P-valor
Intercept                            2.1892   0.0000
C(contact)[T.telephone]              0.8514   0.0531
C(contact)[T.unknown]                0.3696   0.0000
C(perfil_deuda)[T.1]                 0.4869   0.0000
C(poutcome)[T.no_campaign]           0.8636   0.0240
C(poutcome)[T.other]                 1.2933   0.0166
C(poutcome)[T.success]               8.4832   0.0000
age                                  0.9979   0.2559
campaign                             0.9044   0.0000


In [54]:
# 3. CAMINO 1: Agregamos interacciones usando el símbolo '*'
# contact * age -> Evalúa si el canal funciona diferente según la edad
# contact * perfil_deuda -> Evalúa si el canal funciona diferente si el cliente tiene deudas
formula = "deposit ~ C(contact) * age + C(contact) * C(perfil_deuda) + C(poutcome) + campaign"

modelo_interaccion = smf.logit(formula, data=df_marketing).fit()

# 4. Mostrar el nuevo resumen para revisar el Pseudo R-squared
print(modelo_interaccion.summary())

Optimization terminated successfully.
         Current function value: 0.599467
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                deposit   No. Observations:                10987
Model:                          Logit   Df Residuals:                    10974
Method:                           MLE   Df Model:                           12
Date:                Thu, 04 Jun 2026   Pseudo R-squ.:                  0.1343
Time:                        10:24:12   Log-Likelihood:                -6586.3
converged:                       True   LL-Null:                       -7608.0
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
Intercept                               

In [55]:
# 5. Calcular los nuevos Odds Ratios
resultados = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo_interaccion.params),
    'P-valor': modelo_interaccion.pvalues
})
print("\n--- NUEVOS ODDS RATIOS CON INTERACCIONES ---")
print(resultados.round(4))


--- NUEVOS ODDS RATIOS CON INTERACCIONES ---
                                              Odds Ratio (OR)  P-valor
Intercept                                              2.4439   0.0000
C(contact)[T.telephone]                                0.2987   0.0001
C(contact)[T.unknown]                                  0.4067   0.0011
C(perfil_deuda)[T.1]                                   0.4443   0.0000
C(poutcome)[T.no_campaign]                             0.8478   0.0116
C(poutcome)[T.other]                                   1.2903   0.0182
C(poutcome)[T.success]                                 8.2954   0.0000
C(contact)[T.telephone]:C(perfil_deuda)[T.1]           0.9502   0.7761
C(contact)[T.unknown]:C(perfil_deuda)[T.1]             1.9058   0.0000
age                                                    0.9966   0.1129
C(contact)[T.telephone]:age                            1.0216   0.0001
C(contact)[T.unknown]:age                              0.9867   0.0240
campaign                       

In [57]:
# 3. NUEVO (Camino 2): Tramos de 'balance' ADAPTADOS a tus datos reales
condiciones = [
    (df_marketing['balance'] <= 0),
    (df_marketing['balance'] > 0) & (df_marketing['balance'] <= 556),
    (df_marketing['balance'] > 556) & (df_marketing['balance'] <= 2000),
    (df_marketing['balance'] > 2000)
]
opciones = ['1. Negativo o Cero', '2. Saldo Bajo', '3. Saldo Medio', '4. Saldo Alto']
df_marketing['tramo_balance'] = np.select(condiciones, opciones, default='2. Saldo Bajo')

In [61]:
# 4. MODELO DEFINITIVO COMBINADO
formula = """
deposit ~ C(contact) * age 
         + C(contact) * C(perfil_deuda) 
         + C(tramo_balance) 
         + C(poutcome) 
         + campaign
"""

modelo_definitivo = smf.logit(formula, data=df_marketing).fit()

# 5. Ver el resumen para verificar el nuevo Pseudo R-squared de McFadden
print(modelo_definitivo.summary())

Optimization terminated successfully.
         Current function value: 0.595100
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                deposit   No. Observations:                10987
Model:                          Logit   Df Residuals:                    10971
Method:                           MLE   Df Model:                           15
Date:                Thu, 04 Jun 2026   Pseudo R-squ.:                  0.1406
Time:                        10:32:09   Log-Likelihood:                -6538.4
converged:                       True   LL-Null:                       -7608.0
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
Intercept                               

In [62]:
# 6. Calcular los Odds Ratios finales para tu presentación
resultados_finales = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo_definitivo.params),
    'P-valor': modelo_definitivo.pvalues
})
print("\n--- RESULTADOS FINALES COMBINADOS (CAMINO 1 + 2) ---")
print(resultados_finales.round(4))


--- RESULTADOS FINALES COMBINADOS (CAMINO 1 + 2) ---
                                              Odds Ratio (OR)  P-valor
Intercept                                              1.8434   0.0000
C(contact)[T.telephone]                                0.2900   0.0001
C(contact)[T.unknown]                                  0.4093   0.0013
C(perfil_deuda)[T.1]                                   0.4634   0.0000
C(tramo_balance)[T.2. Saldo Bajo]                      1.1620   0.0290
C(tramo_balance)[T.3. Saldo Medio]                     1.5002   0.0000
C(tramo_balance)[T.4. Saldo Alto]                      1.8300   0.0000
C(poutcome)[T.no_campaign]                             0.8701   0.0346
C(poutcome)[T.other]                                   1.2931   0.0179
C(poutcome)[T.success]                                 8.2834   0.0000
C(contact)[T.telephone]:C(perfil_deuda)[T.1]           0.9318   0.6954
C(contact)[T.unknown]:C(perfil_deuda)[T.1]             1.8809   0.0000
age                    

In [64]:
# MODELO CON TODAS LAS INTERACCIONES RESPECTO AL CANAL
formula = """
deposit ~ C(contact) * age 
         + C(contact) * C(perfil_deuda) 
         + C(contact) * C(tramo_balance)
         + C(poutcome) 
         + campaign
"""

modelo_maximo = smf.logit(formula, data=df_marketing).fit()
print(modelo_maximo.summary())

Optimization terminated successfully.
         Current function value: 0.594737
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                deposit   No. Observations:                10987
Model:                          Logit   Df Residuals:                    10965
Method:                           MLE   Df Model:                           21
Date:                Thu, 04 Jun 2026   Pseudo R-squ.:                  0.1411
Time:                        10:42:31   Log-Likelihood:                -6534.4
converged:                       True   LL-Null:                       -7608.0
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------------------
Intercept   

In [66]:
resultados_finales = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo_maximo.params),
    'P-valor': modelo_maximo.pvalues
})
print("\n--- RESULTADOS FINALES COMBINADOS (CAMINO 1 + 2 + 3) ---")
print(resultados_finales.round(4))


--- RESULTADOS FINALES COMBINADOS (CAMINO 1 + 2 + 3) ---
                                                    Odds Ratio (OR)  P-valor
Intercept                                                    1.8074   0.0000
C(contact)[T.telephone]                                      0.2857   0.0027
C(contact)[T.unknown]                                        0.4378   0.0078
C(perfil_deuda)[T.1]                                         0.4646   0.0000
C(tramo_balance)[T.2. Saldo Bajo]                            1.1908   0.0271
C(tramo_balance)[T.3. Saldo Medio]                           1.4932   0.0000
C(tramo_balance)[T.4. Saldo Alto]                            1.8891   0.0000
C(poutcome)[T.no_campaign]                                   0.8727   0.0388
C(poutcome)[T.other]                                         1.2995   0.0160
C(poutcome)[T.success]                                       8.2989   0.0000
C(contact)[T.telephone]:C(perfil_deuda)[T.1]                 0.9280   0.6823
C(contact)[T.unkno